# 예제 04. 가장 단순한 에이전트 만들기

전산실험 2 · AI Agent 만들기 · **1주차** · 예상 소요 20분

## 목표
- LLM 호출과 에이전트의 차이를 코드로 구분한다
- **목표 · 루프 · 종료조건** 세 가지를 붙여 에이전트를 만든다
- 모델 출력을 파싱하다 실패하는 경험을 한다 (3주차 예고)
- 실행 과정을 추적(trace)해서 기록한다 (6주차 예고)

## 오늘 만들 것
```
목표: 1~100 사이의 숨겨진 숫자를 맞혀라

while 안 끝났으면:
    추측 = 모델에게 물어본다(지금까지의 기록)
    결과 = 채점한다(추측)          # "더 큼" / "더 작음" / "정답"
    기록에 추가한다
    if 정답 or 시도 > 10: 멈춘다
```

In [ ]:
!pip install -q openai

import os
from openai import OpenAI

def get_secret(name, secret=True):
    # Colab Secrets -> 환경변수 -> 직접 입력. 값은 코드에 적지 않는다.
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v.strip()
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name].strip()
    if secret:
        import getpass
        v = getpass.getpass(f"{name}: ").strip()
    else:
        v = input(f"{name}: ").strip()
    os.environ[name] = v
    return v


def get_base_url():
    url = get_secret("KMU_BASE_URL", secret=False).rstrip("/")
    for suffix in ("/chat/completions", "/completions"):
        if url.endswith(suffix):
            url = url[: -len(suffix)]
    return url


BASE_URL = get_base_url()
MODEL    = "gemma4"

client = OpenAI(api_key=get_secret("KMU_API_KEY"), base_url=BASE_URL, timeout=120)

def ask(prompt, system=None, **kw):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(model=MODEL, messages=messages, **kw)
    return r.choices[0].message.content.strip()

print("연결 준비 완료:", MODEL)

---
## 1. 에이전트가 아닌 것

지금까지 우리가 한 것은 **한 번 묻고 한 번 답받기** 입니다.
목표도 없고, 반복도 없고, 자기 행동의 결과를 보지도 않습니다.

In [ ]:
# 이것은 에이전트가 아니다 -- 그냥 함수 호출이다
print(ask("1부터 100 사이의 숫자 하나를 고르세요. 숫자만 답하세요.", max_tokens=10))

---
## 2. 부품 1 — 환경 (채점기)

에이전트가 행동하면 반응하는 **바깥 세계**가 필요합니다.
여기서는 숫자를 채점해 주는 아주 단순한 환경을 만듭니다.

> 5주차에 배울 **도구(Tool)** 가 바로 이 자리에 들어갑니다.
> 오늘은 도구가 하나뿐이고 고정된 셈입니다.

In [ ]:
import random

class 숫자게임:
    def __init__(self, 최소=1, 최대=100, seed=None):
        rng = random.Random(seed)
        self.정답 = rng.randint(최소, 최대)
        self.최소, self.최대 = 최소, 최대
        self.시도횟수 = 0

    def 채점(self, 추측):
        self.시도횟수 += 1
        if 추측 == self.정답:
            return "정답"
        return "더 큰 수" if 추측 < self.정답 else "더 작은 수"


게임 = 숫자게임(seed=0)
print("채점기 테스트:", 게임.채점(50), "(정답은", 게임.정답, ")")

---
## 3. 부품 2 — 모델 출력에서 숫자 뽑아내기

여기서 첫 번째 벽에 부딪힙니다.
모델은 `42` 라고만 답하지 않습니다. 이렇게 답할 수도 있습니다.

- `42입니다.`
- `제 추측은 42입니다. 왜냐하면...`
- `범위의 중간인 50을 먼저 시도해 보겠습니다.`

**정규표현식으로 숫자를 긁어내는 임시방편**을 씁니다.
이런 땜질이 왜 위험한지는 3주차에 이야기합니다.

In [ ]:
import re

def 숫자_뽑기(텍스트, 최소=1, 최대=100):
    후보들 = [int(n) for n in re.findall(r"\d+", 텍스트)]
    유효 = [n for n in 후보들 if 최소 <= n <= 최대]
    return 유효[0] if 유효 else None


# 모델이 낼 법한 답들로 시험
for 예시 in ["42", "42입니다.", "제 추측은 42입니다.", "잘 모르겠습니다.",
             "이전에 30, 60을 시도했으니 45를 시도하겠습니다."]:
    print(f"{숫자_뽑기(예시)!s:>6}  <-  {예시!r}")

> 마지막 예시를 보세요. 모델이 과거 기록을 언급하면 **엉뚱한 숫자를 뽑아냅니다.**
> "첫 번째 숫자"를 고르는 규칙이 틀린 것입니다.
>
> 이것이 텍스트를 정규표현식으로 파싱할 때 생기는 전형적인 문제입니다.
> 3주차에는 모델에게 **구조화된 형식**으로 답하게 만들어 이 문제를 해결합니다.

---
## 4. 부품 3 — 루프

이제 세 가지를 붙입니다: **목표**, **반복**, **종료조건**.

In [ ]:
def 숫자맞히기_에이전트(게임, 최대시도=10, verbose=True, **kw):
    기록 = []          # (추측, 결과) 목록 -- 이것이 에이전트의 '기억'이다
    추적 = []          # 실행 로그

    system = (
        "너는 숫자 맞히기 게임을 하고 있다. "
        "1부터 100 사이의 숨겨진 정수를 최소한의 시도로 맞혀야 한다. "
        "지금까지의 힌트를 보고 다음에 시도할 숫자를 하나만 답하라. "
        "반드시 숫자만 답하고 다른 말은 하지 마라."
    )

    for 회차 in range(1, 최대시도 + 1):
        # --- 생각: 지금까지의 기록을 모델에게 보여준다 ---
        if 기록:
            힌트 = "\n".join(f"{g}을(를) 시도했더니 '{r}' 였다." for g, r in 기록)
            프롬프트 = f"지금까지의 결과:\n{힌트}\n\n다음에 시도할 숫자는?"
        else:
            프롬프트 = "첫 번째로 시도할 숫자는?"

        원문 = ask(프롬프트, system=system, max_tokens=20, **kw)
        추측 = 숫자_뽑기(원문)

        # --- 파싱 실패 처리 ---
        if 추측 is None:
            추적.append({"회차": 회차, "원문": 원문, "추측": None, "결과": "파싱실패"})
            if verbose:
                print(f"[{회차}] 파싱 실패: {원문!r}")
            continue

        # --- 행동: 환경에 작용 ---
        결과 = 게임.채점(추측)
        기록.append((추측, 결과))
        추적.append({"회차": 회차, "원문": 원문, "추측": 추측, "결과": 결과})

        if verbose:
            print(f"[{회차}] 추측 {추측:>3}  ->  {결과}")

        # --- 종료조건 ---
        if 결과 == "정답":
            return {"성공": True, "시도": 회차, "추적": 추적}

    return {"성공": False, "시도": 최대시도, "추적": 추적}

In [ ]:
게임 = 숫자게임(seed=7)
print("(정답:", 게임.정답, ")")
print()

결과 = 숫자맞히기_에이전트(게임)
print()
print("성공 여부:", 결과["성공"], "| 시도 횟수:", 결과["시도"])

---
## 5. 무엇을 관찰했는가

여러 번 돌려 보면서 다음을 확인하세요.

| 관찰 | 이것이 예고하는 주차 |
|---|---|
| 모델이 이분탐색을 잘 하기도, 못 하기도 한다 | 3주차 프롬프트 설계 |
| 이미 배제된 범위의 숫자를 다시 부른다 | 4주차 메모리 |
| 숫자 대신 설명을 늘어놓아 파싱이 실패한다 | 3주차 구조화 출력 |
| 매번 결과가 다르다 | 11주차 평가 |
| 기록을 매번 통째로 다시 보낸다 | 4주차 컨텍스트 관리 |

**여러 번 돌려서 성공률을 재 봅시다.** 한 번 성공은 성공이 아닙니다.

In [ ]:
import pandas as pd

결과들 = []
for i in range(10):
    게임 = 숫자게임(seed=i)
    r = 숫자맞히기_에이전트(게임, verbose=False)
    결과들.append({"seed": i, "정답": 게임.정답,
                   "성공": r["성공"], "시도": r["시도"]})
    print(".", end="", flush=True)
print(" 완료")

성적 = pd.DataFrame(결과들)
print()
print(성적)
print()
print(f"성공률: {성적['성공'].mean():.0%}")
print(f"성공한 경우 평균 시도: {성적.loc[성적['성공'], '시도'].mean():.1f}회")
print(f"참고 - 이분탐색의 이론적 최대: {7}회 (log2(100) ≈ 6.6)")

---
## 6. 에이전트의 정의, 다시 보기

```
에이전트 = 모델 + 루프 + 도구 + 목표/종료조건
             ↑      ↑      ↑         ↑
           ask()  for문   채점()   정답 or 10회
```

방금 만든 것이 정확히 이 구조입니다. 앞으로 12주 동안 각 자리를 키워 나갑니다.

- **모델** → 프롬프트를 정교하게, 출력을 구조화 (3주차)
- **루프** → Thought/Action/Observation 을 갖춘 ReAct 엔진 (6주차)
- **도구** → 채점기 하나가 아니라 통계 도구 여러 개 + 레지스트리 (5주차)
- **목표** → 사용자의 자연어 질문을 목표로 변환 (9주차)

---
## 연습문제

**문제 1.** `최대시도` 를 5로 줄이면 성공률이 얼마나 떨어지는가?
각 조건에서 20회씩 돌려 비교하고, 두 비율의 차이를 검정하시오.

**문제 2.** `system` 프롬프트에 "이분탐색 전략을 사용하라: 가능한 범위의 중간값을
시도하라" 를 추가하면 평균 시도 횟수가 줄어드는가? 20회씩 돌려 비교하시오.

**문제 3.** `숫자_뽑기()` 는 "이전에 30, 60을 시도했으니 45를..." 같은 문장에서
엉뚱한 값을 뽑아냅니다. 이 함수를 개선할 수 있겠는가?
개선해도 남는 근본적 한계는 무엇인가?

**문제 4.** 파싱에 실패했을 때 지금은 그냥 넘어갑니다.
모델에게 "숫자만 답하라"고 다시 요구하는 **재시도 로직**을 넣어 보시오.

In [ ]:
# 여기에 직접 작성해 보세요

<details>
<summary><b>해설 보기</b></summary>

**문제 1.**
이분탐색이 완벽하다면 7회면 충분하므로, 5회 제한은 이론적으로도 빠듯합니다.
성공률이 크게 떨어집니다. 두 비율 비교에는 `scipy.stats.fisher_exact` 또는
`statsmodels.stats.proportion.proportions_ztest` 를 씁니다. n=20 이면
검정력이 낮으니 결론을 조심스럽게 내리십시오.

**문제 2.**
대개 개선됩니다. **전략을 프롬프트로 알려주는 것**만으로 성능이 오르는데,
이것이 프롬프트 엔지니어링의 핵심입니다(3주차).
다만 소형 모델은 지시해도 중간값 계산을 자주 틀립니다.
→ 그래서 5주차에 **계산은 도구에게 맡기는** 방식을 배웁니다.

**문제 3.**
"마지막 숫자를 고른다", "'다음은'/'시도하겠습니다' 뒤의 숫자를 고른다" 등으로
개선할 수 있지만 전부 땜질입니다.
**근본 한계**: 자유 텍스트에서 의도를 추출하는 것은 원리적으로 애매합니다.
해법은 파서를 정교하게 만드는 것이 아니라 **모델의 출력 형식 자체를 고정**하는 것입니다.
`{"guess": 42}` 같은 JSON으로 받으면 애매함이 사라집니다. → 3주차

**문제 4.**
```python
for 재시도 in range(3):
    원문 = ask(프롬프트, system=system, max_tokens=20)
    추측 = 숫자_뽑기(원문)
    if 추측 is not None:
        break
    프롬프트 = "숫자 하나만 답하세요. 설명하지 마세요."
```
이 "실패하면 오류를 알려주고 다시 요구하는" 패턴이
3주차 `ask_json()` 과 6주차 ReAct 루프의 핵심 아이디어입니다.

</details>

---
## 1주차 정리

오늘 배운 것

1. 에이전트 = **모델 + 루프 + 도구 + 종료조건**
2. Chat API는 메시지 목록을 주고받는다. **모델은 기억하지 못한다**
3. LLM은 **확률적 시스템**이다. 한 번 성공은 성공이 아니다
4. 소형 모델은 **형식 지시를 자주 어긴다** — 이것이 앞으로의 주된 싸움이다
5. API 키는 절대 코드에 적지 않는다

## 다음 주 예고

오늘 우리는 같은 코드(`get_key`, `ask`, `ask_retry`)를 노트북마다 복사해 붙였습니다.
불편했을 것입니다. 다음 주에는

- Git으로 코드의 변경 이력을 관리하고
- GitHub에 자신의 저장소를 만들고
- 이 코드들을 `kmu_llm.py` 파일 하나로 정리해 올린 뒤
- 새 Colab 런타임에서 내려받아 `from kmu_llm import ask` 로 쓰는 데까지

해 보겠습니다. 앞으로 12주 동안 그 저장소가 여러분의 작업 공간이 됩니다.